# Kieker Explorations

imports

In [3]:
import networkx as nx

from kieker_analysis.charts.radar import run_and_merge
from kieker_analysis.graph.layouts.agent_activity import layout_agent_events
from kieker_analysis.graph.layouts.util import collapse_nodes, make_cyto_from_nx
from util import get_event_name
from kieker_analysis.queries.queries import QueryOld
from kieker_analysis.sparql.client import run_dataframe
from ipycytoscape import CytoscapeWidget
from kieker_analysis.graph.cyto_styles import GraphStyle
from kieker_analysis.graph.graphs import issue_graph, user_role_graph, timed_user_role_graph
from time import sleep

In [5]:


nx_graph = user_role_graph(user)
viz_graph = nx_graph.copy()
collapse_nodes(viz_graph)
layout_agent_events(viz_graph,user)

In [6]:
cyto=make_cyto_from_nx(viz_graph,GraphStyle.usr_role.value)
display(cyto)

CytoscapeWidget(cytoscape_layout={'name': 'preset', 'animate': False}, cytoscape_style=[{'selector': 'node', '…

# user's activity types

In [ ]:
# set this to see the graph for an issue or PR
issue="https://github.com/kieker-monitoring/kieker/pull/103"
cyto=CytoscapeWidget()
graph = issue_graph(issue)
cyto.graph.add_graph_from_networkx(graph)
cyto.set_style(GraphStyle.default.value)
cyto.layout.width = "100%"
cyto.layout.height = "100%"
cyto.set_layout(
    name="breadthfirst",
    #directed=True,
     spacingFactor=1.2,
    # padding=10
)

display(cyto)

# user's activity over time
operate the slider to advance

In [ ]:
# user = "shinhyungyang"
user = "madcau"
# user = "avanhoorn"
time_graphs: list[nx.DiGraph]=[]

for nxg in timed_user_role_graph(user, iters=10, step=1):
    # print (nxg.nodes)
    collapse_nodes(nxg,id_prefix="events")
    layout_agent_events(nxg,user)
    time_graphs.append(nxg)
print(len(time_graphs))

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def show_time_graph(i: int):
    nxg = time_graphs[i]
    cyto_t = make_cyto_from_nx(nxg, GraphStyle.usr_role.value)
    display(cyto_t)

slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(time_graphs) - 1,
    step=1,
    description="Slice",
    continuous_update=False,
)

widgets.interact(show_time_graph, idx=slider)

## Collaboration

## Misc

In [ ]:
from kieker_analysis.charts.radar import run_and_merge
import pandas as pd
import plotly.express as px
df=run_and_merge()


In [ ]:

# user = "shinhyungyang"
# user = "madcau"
# user = "avanhoorn"
usern = "DanielMSchmidt"
# usern = "Marcel-Ha"
user = f"https://github.com/{user}"

print(user)

metric_cols = ["interactivity", "variety", "pulls", "comments"]
agent_row = df[df["agent"] == user]

long = agent_row.melt(
    id_vars="agent",
    value_vars=metric_cols,
    var_name="metric",
    value_name="value",
)

fig = px.line_polar(
    long,
    r="value",
    theta="metric",
    line_close=True,
    title=user,
)

# Fill the polygon
fig.update_traces(fill="toself")

# Hide axis labels (tick labels)
fig.update_layout(
    polar=dict(
        radialaxis=dict(showticklabels=False, ticks=""),
        angularaxis=dict(showticklabels=True, ticks=""),
    )
)


fig.show()



How collaborative are contributors within the project?

How much inter-organisation work is there?

organisation - contributor - graph

How many other repos do people work on?

see what an issue looks like as a network.

table: changes to issues by type


The roles of a user (graph)
 assign a user to the ```user``` variable to see the graph for that user...

In [ ]:
df = run_dataframe(QueryOld.issues_events)
df = df.pivot_table(
    index = "issue",
    columns = "event_type",
    values = "count",
    fill_value = 0
)
df.columns = df.columns.map(get_event_name)
df.drop(columns="prov#Activity", inplace=True)
event_type_totals = (df.sum().rename("total").reset_index() )
display(event_type_totals)

table: contributors, number of issue versions per agent

In [ ]:
df = run_dataframe(QueryOld.contributor_activity)
df.sort_values(by="issues", ascending=False)

How many users have an ORCID in their GitHub profile?

How many users provide affilation details in their profile?


how many contributors share a name with the author of a Kieker publication

what github organisations do contributors work for?